# 🎙️ Text to Speech (TTS) & Speech-to-Text (STT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-hands-on/blob/main/2_Multimodal_Generation/03-TTS_STT-Lecture.ipynb)

> **주의:** Colab 런타임을 재시작하면 `OPENAI_API_KEY`를 다시 설정해야 합니다.

## 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U "openai[realtime]"

In [1]:
# Colab에서 실행 중이라면 실습 데이터(data/)가 있는 레포를 받아옵니다.
# (로컬에서 레포를 clone 해서 실행 중이라면 그냥 넘어갑니다)
import os, sys

if "google.colab" in sys.modules and not os.path.exists("data"):
    !git clone -q --depth 1 https://github.com/nawooooojin/openai-api-hands-on.git /content/openai-api-hands-on
    %cd /content/openai-api-hands-on/2_Multimodal_Generation

### API 키 설정 및 Client 생성
- **Colab**: 왼쪽 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장하고 **노트북 액세스**를 켜 두면 자동으로 불러옵니다.
- **로컬**: 환경 변수 `OPENAI_API_KEY`를 미리 설정해 두거나, 아래 셀을 실행했을 때 뜨는 입력창에 키를 붙여넣으면 됩니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [2]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [3]:
base_dirs = [
    ("data", ["image", "audio", "video"]),
    ("results", ["image", "audio", "video"]),
]

for root, subfolders in base_dirs:
    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

ready: data/image
ready: data/audio
ready: data/video
ready: results/image
ready: results/audio
ready: results/video


In [4]:
from openai import OpenAI
from pathlib import Path
from IPython.display import Audio, display

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


## Text-to-Speech (TTS)

TTS는 텍스트를 오디오로 변환하는 모델입니다.

OpenAI API에는 13개의 내장 음성이 포함되어 있으며 다음과 같이 사용할 수 있습니다:
- 작성된 블로그 포스트를 나레이션하기
- 다국어로 음성 오디오 생성하기
- 스트리밍을 사용한 실시간 오디오 출력 제공하기

> ⚠️ **2026년 10월 업데이트**: Speech API의 TTS 모델(`gpt-4o-mini-tts`, `tts-1`, `tts-1-hd`)은 deprecated 되어 **2027년 1월 6일에 종료**될 예정입니다.
> OpenAI가 안내하는 대체 모델은 Realtime API의 `gpt-realtime-2.1-mini`라서, 이 노트북 TTS 파트 마지막에 **Realtime API로 음성 만드는 버전**도 같이 넣어두었습니다.

### 주요 파라미터

**model**: 사용 가능한 TTS 모델 중 하나를 지정합니다.
- `gpt-4o-mini-tts`, `tts-1`, `tts-1-hd` 중 선택 가능합니다. (셋 다 2027.01.06 종료 예정)

**input**: 오디오를 생성할 텍스트입니다.
- 최대 길이는 4096자입니다.

**voice**: 오디오를 생성할 때 사용할 음성입니다.
- 지원되는 음성은 `alloy`, `ash`, `ballad`, `coral`, `echo`, `fable`, `nova`, `onyx`, `sage`, `shimmer`, `verse`, `marin`, `cedar` 입니다.
- 가장 품질이 좋은 음성으로 `marin`, `cedar`가 추천됩니다.
- 음성의 미리듣기는 [OpenAI.fm](https://openai.fm)에서 확인할 수 있습니다.

**instructions**: 생성된 오디오의 음성을 추가 지시사항으로 제어합니다.
- 억양, 감정, 속도 등을 지정할 수 있습니다.
- `tts-1` 또는 `tts-1-hd` 모델에서는 작동하지 않습니다.

**response_format**: 오디오 출력 형식을 지정합니다.
- 지원되는 형식은 `mp3`(기본값), `opus`, `aac`, `flac`, `wav`, `pcm` 입니다.

**speed**: 생성된 오디오의 속도입니다.
- 0.25에서 4.0 사이의 값을 선택할 수 있습니다.
- 기본값은 1.0입니다.

In [5]:
from pathlib import Path
from openai import OpenAI

client = OpenAI()
speech_file_path = "results/audio/tts_sample.mp3"


with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="marin",
    input="와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!",
    instructions="Speak in a cheerful and positive tone.",
) as response:
    response.stream_to_file(speech_file_path)

print("✅ 저장 완료 ->", speech_file_path)
display(Audio(speech_file_path))

✅ 저장 완료 -> results/audio/tts_sample.mp3


### Realtime API로 음성 만들기 (`gpt-realtime-2.1-mini`)

Speech API 종료 이후를 대비해서, Realtime API로 같은 문장을 읽게 해 보겠습니다.

**Speech API와 다른 점**
- HTTP 요청 한 번이 아니라, **WebSocket으로 연결**해서 이벤트를 주고받는 방식입니다.
- 원래는 음성 대화(speech-to-speech)용 API라서, `instructions`로 *"텍스트를 그대로 읽어라"* 라고 TTS 역할을 지정해 줍니다.

**흐름**
1. `client.realtime.connect(model=...)`로 세션 연결
2. `session.update`로 출력 형식(오디오), 음성(voice), 지시사항 설정
3. `conversation.item.create`로 읽을 텍스트 전달 → `response.create`로 응답 요청
4. `response.output_audio.delta` 이벤트로 들어오는 오디오 조각(PCM 16bit, 24kHz)을 모아서 WAV로 저장

In [6]:
import base64
import wave

text = "와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!"
audio_chunks = bytearray()

with client.realtime.connect(model="gpt-realtime-2.1-mini") as conn:
    # 1) 세션 설정: 오디오로만 출력 + 음성 + TTS 역할 지시
    conn.session.update(session={
        "type": "realtime",
        "output_modalities": ["audio"],
        "instructions": (
            "You are a text-to-speech engine. Read the user's text aloud exactly as written, "
            "in a cheerful and positive tone. Do not add or omit any words."
        ),
        "audio": {"output": {"voice": "marin", "format": {"type": "audio/pcm", "rate": 24000}}},
    })

    # 2) 읽을 텍스트 전달 + 응답 요청
    conn.conversation.item.create(item={
        "type": "message",
        "role": "user",
        "content": [{"type": "input_text", "text": text}],
    })
    conn.response.create()

    # 3) 이벤트를 받으면서 오디오 조각 모으기
    for event in conn:
        if event.type == "response.output_audio.delta":
            audio_chunks.extend(base64.b64decode(event.delta))
        elif event.type == "response.output_audio_transcript.done":
            print("🗣️ 읽은 문장:", event.transcript)
        elif event.type == "error":
            print("❌ 오류:", event.error)
            break
        elif event.type == "response.done":
            break

# 4) PCM → WAV 저장 (mono, 16bit, 24kHz)
realtime_file_path = "results/audio/tts_realtime_sample.wav"
with wave.open(realtime_file_path, "wb") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(24000)
    wf.writeframes(bytes(audio_chunks))

print("✅ 저장 완료 ->", realtime_file_path)
display(Audio(realtime_file_path))

🗣️ 읽은 문장: 와! 오늘 같은 날은 LLM 실습 배우기에 완전히 최고의 날이에요! 정말 신나요!


✅ 저장 완료 -> results/audio/tts_realtime_sample.wav


## Speech-to-Text (STT)

**Speech-to-Text** 는 사람의 말소리를 텍스트로 변환하는 기술입니다.

**사용 예시**
- 오디오의 언어 그대로 텍스트로 옮기는 전사 (transcription)
- 오디오를 영어로 번역 + 전사 작업
- 실시간 자막 생성
- 음성 메모 변환

**OpenAI Audio API는 두 가지 speech-to-text 엔드포인트를 제공합니다.**
- **`transcriptions`**: 오디오를 해당 언어의 텍스트로 변환
- **`translations`**: 오디오를 영어 텍스트로 번역 및 변환 (⚠️ 2027.02.26 종료 예정)

**지원 모델 (2026년 10월 기준)**
- **`gpt-transcribe`**: 녹음된 파일 전사용 추천 모델
- **`gpt-live-transcribe`**: 실시간(저지연) 전사용 모델
- ~~`whisper-1`, `gpt-4o-transcribe`, `gpt-4o-mini-transcribe`, `gpt-4o-transcribe-diarize`~~: 2025년 수업 때 쓰던 모델들로, deprecated 되어 **2027년 2월 26일 종료** 예정


**파일 제한사항**
- **파일 크기**: 최대 25MB
- **지원 형식**: `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `wav`, `webm`

**실습 오디오**
- `data/audio/`의 음성 4개는 신조어·줄임말이 들어간 일상 대화 문장을 TTS로 직접 만든 파일입니다.

In [7]:
for name in ["lol", "convenience_store", "gasimbi", "doseguon"]:
    print(name)
    display(Audio(f"data/audio/{name}.mp3"))

lol


convenience_store


gasimbi


doseguon


## 1) Transcriptions

### 기본 흐름
- `client.audio.transcriptions.create`로 오디오 파일과 모델을 지정해 텍스트를 얻습니다.
- 실습 모델: `gpt-transcribe`
- 응답은 기본적으로 JSON(`response.text`)이며, `response_format="text"`로 순수 텍스트를 받을 수 있습니다.

### 주요 파라미터
- **`file`**: 변환할 오디오 파일 객체 (파일 이름이 아님)
- **`model`**: 사용할 모델 (`gpt-transcribe` 등)
- **`languages`**: 입력 오디오의 언어 힌트 리스트 (예: `["ko"]`, `["ko", "en"]`)
    - ISO-639-1 형식 코드 사용: [ISO-639-1 형식](https://en.wikipedia.org/wiki/List_of_ISO_639_language_codes)
    - `gpt-transcribe`에서는 예전의 단수형 `language` 대신 리스트형 `languages`를 씁니다. (둘을 같이 보내면 안 됨)
- **`prompt`**: (선택) 대화 주제, 맥락 등 자유 형식의 힌트
- **`keywords`**: (선택) 잘 틀리는 고유명사·전문용어 리스트 (`gpt-transcribe`에서 새로 생긴 파라미터)
- **`response_format`**: 출력 형식 (`gpt-transcribe`는 `json`, `text` 지원)

In [8]:
from openai import OpenAI

client = OpenAI()

with open("data/audio/gasimbi.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
    )

print(transcription.text)

난 몇 달 전만 해도 가심비가 가성비보다 중요하다고 생각해서 뭐 살 때 그리 고민하지 않았는데 요즘은 달라.


**기본 응답 타입 (json) 예시**
- 전사 텍스트(`text`)와 함께 감지된 언어(`languages`), 사용량(`usage`) 정보가 들어 있습니다.

In [9]:
print(transcription.model_dump_json(indent=2))

{
  "text": "난 몇 달 전만 해도 가심비가 가성비보다 중요하다고 생각해서 뭐 살 때 그리 고민하지 않았는데 요즘은 달라.",
  "languages": [
    {
      "code": "ko"
    }
  ],
  "logprobs": null,
  "usage": {
    "seconds": 10.0,
    "type": "duration"
  }
}


## 2) Translation

### (기존 방식) translations 엔드포인트
- `client.audio.translations.create`로 오디오 파일을 입력받아 영어 텍스트로 번역 및 전사합니다.
- `whisper-1` 모델만 지원하고, 출력은 원본 언어가 아닌 **영어 텍스트**로 변환됩니다.
- ⚠️ `whisper-1`과 `/v1/audio/translations`는 **2027년 2월 26일 종료** 예정입니다.
    - 실시간 번역은 Realtime API의 `gpt-realtime-translate`로 옮겨가는 중입니다.

In [10]:
from openai import OpenAI

client = OpenAI()

# ⚠️ whisper-1 / translations 엔드포인트는 2027-02-26 종료 예정
with open("data/audio/convenience_store.mp3", "rb") as audio_file:
    translation = client.audio.translations.create(
        model="whisper-1",
        file=audio_file,
    )

print(translation.text)

That's right. Sometimes, there are places where a lot of good customers come. In the case of convenience stores, there are people who give one plus one product each.


### (추천) 전사 → GPT 번역 2단계
- 엔드포인트 종료와 상관없이 쓸 수 있는 방법: `gpt-transcribe`로 받아쓰고, Responses API로 번역합니다.
- 영어뿐 아니라 **원하는 어떤 언어로도** 번역할 수 있고, 말투도 지정할 수 있다는 장점이 있습니다.

In [11]:
with open("data/audio/convenience_store.mp3", "rb") as audio_file:
    korean_text = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        languages=["ko"],
        response_format="text",
    )

response = client.responses.create(
    model="gpt-6-luna",
    instructions="Translate the user's Korean text into natural, casual English. Output only the translation.",
    input=korean_text,
)

print("🇰🇷", korean_text.strip())
print("🇺🇸", response.output_text)

🇰🇷 맞아. 가끔씩 좋은 손님들 많이 오는 지역도 있거든. 편의점의 경우에는 원 플러스 원 상품을 하나씩 주고 가시는 분들도 있더라고.
🇺🇸 Yeah. Some areas get a lot of really nice customers. At convenience stores, some people even leave you one of their buy-one-get-one-free items.


## 3) 고급 옵션: 응답 포맷 · 프롬프트 · 키워드

### 응답 포맷 (response_format)
- **`json`**: 기본 JSON 형식 (텍스트 포함)
- **`text`**: 순수 텍스트만 반환
- 자막 포맷(`srt`, `vtt`)이나 타임스탬프가 들어간 `verbose_json`은 `whisper-1` 전용이었고, `gpt-transcribe`에서는 지원하지 않습니다. (2026년 10월 기준)

### 프롬프트 (prompt) / 키워드 (keywords)
- 모델이 잘못 인식하는 용어/고유명사를 안내해 정확도 향상
- 이전 오디오 세그먼트의 텍스트를 전달해 맥락 유지

### 텍스트 포맷 실습
- 신조어 '도세권'(도서관 + 역세권)이 들어간 문장을 받아써 보겠습니다. 처음 듣는 단어라 틀리게 받아쓸 수도 있습니다.

In [12]:
from openai import OpenAI

client = OpenAI()

with open("data/audio/doseguon.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        response_format="text",
    )

print(transcription)

내가 오늘은 도색권이라는 걸 알게 됐는데, 도서반하고 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.



### Prompting 실습

`prompt`, `keywords` 파라미터를 활용하여 전사 품질을 향상시킬 수 있습니다.

**주요 활용 사례**

- **고유명사/약어/신조어 오인식 개선**:
    - 예: DALL·E, GPT-3가 "GPT 3", "DALI"로 전사될 경우
    - 프롬프트: `"The transcript is about OpenAI which makes model like DALL·E, GPT-3, and ChatGPT..."`
    - 키워드: `["DALL·E", "GPT-3", "ChatGPT"]`

- **세그먼트 간 문맥 유지**:
    - 긴 오디오를 나눠서 전사할 때, 앞 조각의 전사 결과를 다음 조각의 `prompt`로 넘겨주기

- **문장부호 포함 유도**:
    - 예: `"Hello, welcome to my lecture."`

- **말채움(filler) 단어 유지**:
    - 예: `"Umm, let me think like, hmm... Okay, here's what I'm, like, thinking."`

- **표기 체계 지정**:
    - 간체/번체 중국어 등

In [13]:
# prompt로 대화 주제(맥락) 알려주기
with open("data/audio/doseguon.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        response_format="text",
        prompt="이 대화는 신조어에 대한 대화입니다. 도세권은 도서관과 역세권의 합성어로, 도서관이 가까운 곳을 뜻합니다.",
    )

print(transcription)

내가 오늘은 도세권이라는 걸 알게 됐는데, 도서관하고 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.



In [14]:
# keywords로 잘 틀리는 단어 직접 알려주기
with open("data/audio/doseguon.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio_file,
        response_format="text",
        keywords=["도세권", "역세권", "도서관"],
    )

print(transcription)

내가 오늘은 도세권이라는 걸 알게 됐는데 도서관하고 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.



### Post-processing with GPT

전사 품질 향상을 위해 GPT 모델을 활용한 후처리(post-processing)를 수행할 수 있습니다.

- **방법**: Responses API를 사용해 전사된 텍스트를 GPT 모델에 입력하고, 오타 수정 및 문맥 개선을 요청합니다.
- **developer 메시지 활용**: 회사명, 제품명, 신조어 등 용어 사전을 developer 메시지에 넣어 정확도를 높일 수 있습니다.
- **장점**: `prompt` 파라미터와 유사하지만, 더 복잡한 지시사항과 맥락을 제공할 수 있습니다.

In [15]:
system_prompt = """
당신은 일상 대화를 전사하는 도우미입니다. 전사된 텍스트의 오타를 수정하고,
신조어·줄임말이 올바르게 표기되었는지 확인해주세요.
용어 사전: 도세권(도서관 + 역세권), 역세권, 가심비(가격 대비 마음의 만족), 가성비, 주챔(주요 챔피언), 원플러스원.
필요한 구두점(마침표, 쉼표)을 추가하되, 제공된 맥락만 사용하고 교정된 문장만 출력하세요.
"""


def generate_corrected_transcript(audio_file_path, reasoning_effort="low", verbosity="low", system_prompt=system_prompt):
    # 1단계: 오디오를 텍스트로 전사
    with open(audio_file_path, "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="gpt-transcribe",
            file=audio_file,
            response_format="text"
        )

    # 2단계: 전사된 텍스트를 GPT로 후처리
    response = client.responses.create(
        model="gpt-6-luna",
        reasoning={"effort": reasoning_effort},
        text={"verbosity": verbosity},
        input=[
            {
                "role": "developer",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"다음 전사 텍스트를 교정해주세요:\n{transcription}"
            }
        ]
    )

    return response.output_text, transcription

# 신조어 오디오 파일로 STT 및 후처리
audio_file_path = "data/audio/doseguon.mp3"
corrected_text, original_text = generate_corrected_transcript(audio_file_path)

print("원본 전사:")
print(original_text)
print("\n교정된 텍스트:")
print(corrected_text)

원본 전사:
내가 오늘은 도색권이라는 걸 알게 됐는데, 도서반하고 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.


교정된 텍스트:
내가 오늘은 도세권이라는 걸 알게 됐는데, 도서관과 역세권의 합성어로 도서관이 가까운 거리에 있는 곳을 뜻해.


### 참고 자료
- https://platform.openai.com/docs/guides/text-to-speech
- https://platform.openai.com/docs/guides/realtime
- https://platform.openai.com/docs/guides/speech-to-text
- https://platform.openai.com/docs/api-reference/audio
- https://openai.fm
- 실습 오디오 데이터
    - 원래 수업에서는 AI Hub의 「연령대별 특징적 발화(은어·속어 등) 음성 데이터」를 사용했는데, 공개 레포에는 재배포가 어려워서 같은 느낌의 문장을 `gpt-4o-mini-tts`로 직접 만들어 대체했습니다.
    - 원본 데이터가 궁금하다면: [AI Hub](https://aihub.or.kr)